# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [2]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [3]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [5]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the provided information, exercises that can help with lower back pain include:\n\n1. Cat-Cow Stretch: Start on your hands and knees. Alternate between arching your back up (like a cat) and letting it sag down (like a cow). Perform 10-15 repetitions.\n\n2. Bird Dog: From your hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Do 10 repetitions on each side.\n\n3. Pelvic Tilts: Lie on your back with knees bent. Flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises are gentle and specifically targeted to alleviate lower back discomfort and strengthen the supporting muscles. As always, consult with a healthcare professional before starting new exercises, especially if you have ongoing or severe pain.'

In [12]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical restoration, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (7-9 hours per night for adults) is crucial for maintaining a strong immune system, managing stress, and promoting optimal functioning of various body systems. Poor sleep or sleep disorders like insomnia can negatively impact health, leading to issues such as weakened immunity, increased stress levels, and impaired cognitive performance. Therefore, practicing good sleep hygiene and ensuring sufficient rest are essential for overall health and wellness.'

In [13]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gently massaging the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing deep breathing techniques and progressive muscle relaxation\n- Taking short walks, especially in nature\n- Listening to calming music\n- Engaging in mindfulness and meditation practices\n\nThese approaches can help manage stress and alleviate headache symptoms naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [11]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From a hands-and-knees position, extend opposite arm and leg while engaging your core. Hold for about 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises may help alleviate lower back discomfort and prevent future episodes.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health and wellness. Maintaining a consistent sleep schedule and creating an optimal sleep environment—such as a cool, dark, and quiet room—helps improve sleep quality. Good sleep hygiene practices, including a relaxing bedtime routine and limiting screen time before bed, promote restorative sleep. Adequate and quality sleep supports immune function, mental health, mood, energy levels, and overall physical health. Conversely, poor sleep or insomnia can negatively impact these areas, emphasizing the importance of healthy sleep habits for overall well-being.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include relaxation techniques such as deep breathing and meditation, progressive muscle relaxation, herbal teas like chamomile or valerian root, and ensuring proper hydration and good sleep habits. Additionally, practices like gentle stretching, managing stress triggers, and maintaining a healthy diet can help reduce headache frequency and intensity.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

Naive retrieval uses similarity distance metric to retrieve each chunk separately. There is no gurantee that they are all related. The BM25 uses comon words among the chunks and ranks them based on that, which can result in high relevant results.

For example, given the user query: What is the recommended daily intake of Vitamin C?,

BM25 -> looks for the exact phrase "Vitamin C" in the documents. Good for lexical matching.

Embeddings - may return documents related to synonyms and paraphrases as well. It may be useful in some cases, but in this case we want exact lexical matching. 

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [12]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [13]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"To help alleviate lower back pain, gentle stretching and strengthening exercises are recommended. Some specific exercises include:\n\n- **Cat-Cow Stretch:** Starting on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for about 5 seconds, then switch sides. Aim for 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, then repeat 8-12 times.\n\nThese exercises can help improve flexibility and strengthen the muscles supporting your lower back. However, it's always best to consult with a healthcare professional before starting any new exercise routine, especially if you have significant pain or underlying health conditions."

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is crucial for maintaining physical health, mental well-being, and cognitive function. During sleep, your body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Getting enough quality sleep—typically 7-9 hours per night—is essential for supporting these processes and promoting overall wellness.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing, progressive muscle relaxation, grounding techniques, taking a short walk in nature, listening to calming music, staying hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of the temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [14]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [15]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Perform 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises are recommended for alleviating lower back discomfort and preventing future i

In [27]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health by supporting physical restoration, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours for adults—helps strengthen the immune system, manage stress, and improve mood. Poor sleep or sleep disturbances like insomnia can lead to physical and mental health problems, including fatigue, difficulty concentrating, and increased risk of chronic conditions. Maintaining good sleep hygiene, creating a comfortable sleep environment, and establishing consistent routines are essential for ensuring quality sleep and, consequently, better overall health.'

In [28]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking plenty of water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Giving gentle massages to the temples and neck\n- Using essential oils such as peppermint or lavender\n- Engaging in relaxation techniques like deep breathing, progressive muscle relaxation, or grounding exercises\n- Taking short walks, especially in nature\n- Listening to calming music\n\nAdditionally, maintaining a regular sleep schedule, practicing mindfulness and meditation, and implementing a calming evening routine can help manage stress and reduce headache frequency.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:
A single query may miss relevant chunks phrased differently. Through multi-query expansion, we are generating many variants of the same question, each of which can retrieve different chunks. The union of these results represents a more comprehensive list of related documents thus  improving recall.


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [16]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [17]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [18]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [19]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [20]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"I'm happy to help! Some exercises that can help with lower back pain include gentle stretches and strengthening exercises such as:\n\n- Cat-Cow stretches\n- Child's pose\n- Pelvic tilts\n- Knee-to-chest stretches\n- Bridge exercises\n- Bird-Dog exercises\n\nHowever, it's important to consult with a healthcare professional or physical therapist to determine the most appropriate exercises for your specific condition and to ensure you're performing them safely."

In [34]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health. It helps the body repair and regenerate tissues, supports immune function, and regulates hormones. Adequate sleep is essential for mental clarity, mood stability, and cognitive performance. Poor or insufficient sleep can lead to a range of health issues, including weakened immunity, weight gain, increased risk of chronic diseases like heart disease and diabetes, and mental health challenges such as depression and anxiety. Therefore, quality sleep is crucial for maintaining overall physical and mental well-being.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"I'm sorry, but I don't have any information from the provided context to answer your question. However, some common natural remedies for stress and headaches include practices like yoga, meditation, regular exercise, staying hydrated, getting enough sleep, and using herbal teas such as peppermint or chamomile. If you have ongoing or severe symptoms, it's best to consult a healthcare professional."

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [21]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [22]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, then repeat 8-12 times.\n\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises focus on gentle stretching and strengthening, which can alleviate lower back discomfort an

In [40]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical recovery, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (7-9 hours for adults) is linked to a stronger immune system, better mood, improved concentration, and reduced risk of chronic conditions such as headaches, digestive issues, and stress-related ailments. Maintaining good sleep hygiene, such as keeping a consistent schedule, creating a comfortable sleep environment, and practicing relaxing routines before bed, can significantly enhance sleep quality and overall health.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (such as inhaling for 4 counts, holding, exhaling, and holding again)\n- Progressive muscle relaxation, tensing and releasing muscle groups\n- Grounding techniques, like naming things you see, hear, feel, smell, and taste\n- Taking short walks, preferably in nature\n- Listening to calming music\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils such as peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing mindfulness or relaxation exercises\n\nThese approaches can help alleviate stress and reduce the occurrence and intensity of headaches naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [23]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [24]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [25]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [26]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [27]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [48]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Repeat 8-12 times.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises are gentle stretching and strengthening movements recommended for alleviating lower back discomfort and preventing future episodes.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive well-being. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—generally 7-9 hours per night for adults—helps maintain a healthy immune system, improves mood, enhances cognitive function, and reduces the risk of various health conditions. Poor sleep quality or insufficient sleep can negatively impact these processes, leading to issues such as fatigue, weakened immunity, cognitive impairment, and increased risk of chronic diseases. Therefore, prioritizing good sleep hygiene and creating an optimal sleep environment are crucial for maintaining overall health.'

In [50]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated to prevent dehydration-related headaches.\n- Applying cold or warm compresses to the head or neck.\n- Resting in a dark, quiet room to reduce headache triggers.\n- Gentle massage of the temples and neck muscles.\n- Using essential oils such as peppermint or lavender, which may help alleviate headaches.\n- Practicing relaxation techniques like deep breathing, progressive muscle relaxation, or mindfulness meditation.\n- Engaging in short walks or listening to calming music to reduce stress levels.\n\nThese remedies can help manage stress and headaches naturally. However, if symptoms persist, it's best to consult a healthcare provider."

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

FAQs usually have similar sentences (highly repetitive), so semantic distances are small. This means there is low variance in the distance distribution.
This can also make the breakpoints harder to detect reliably.
There may be too many chunks or too few chunks depending on the threshold and the breakpoint method.

Using structure based chunking can help FAQs because they follow specific structure (headings, questions, answers). Since there may not enough variation in the sentence distribution, std dev threshold is not a good idea to use.
Can leverage percentile or interquartile (for better outlier tolerance) threshold methods.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

### Plan

Use RAG for howpeopleuseai.pdf research paper

#### Golden dataset
- Number of test samples: 12
- Use abstracted approach for synthetic data generation
- LLM to use: GPT 4.1

#### Retriever
- Number of chunks to retrieve: 8 (For reranking - 3)
- Parent document size: 2000, overlap 200
- Child document size: 200, overlap 50
- Embedding model: text-embedding-3-small
- LLM: GPT 4o mini
- Retrievers to compare:
   - Naive
   - BM-25
   - Multi-Query Retrieval
   - Parent Document Retriever
   - Ensemble
- Semantic chunking on vs off

#### Metrics
- Faithfulness
- Answer relevance
- Context Recall
- Context Precision
- Context Entity Recall
- Noise Sensitivity

#### Observability
- use Langsmith to measure latency, cost


In [28]:
import os
import getpass

# Enable LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key:")  # or your key directly
os.environ["LANGCHAIN_PROJECT"] = "Advanced-Retrieval-Activity1"  # project name in LangSmith UI

In [29]:

# Naive RAG
 
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain.prompts import ChatPromptTemplate
from qdrant_client import QdrantClient, models


chat_model = ChatOpenAI(model="gpt-4o-mini") 

loader = PyPDFLoader("data/howpeopleuseai.pdf")
raw_docs_ai = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
ai_docs = text_splitter.split_documents(raw_docs_ai)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    ai_docs,
    embeddings,
    location=":memory:",
    collection_name="how_people_use_ai",
)

naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 8})

RAG_TEMPLATE = """
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

naive_retrieval_chain = (
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)


# BM25 RAG
bm25_retriever = BM25Retriever.from_documents(ai_docs, k=8)

bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)


# Multi-Query RAG
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)


# Parent Document RAG

parent_splitter_ai = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter_ai = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)

store_ai = InMemoryStore()
client_ai = QdrantClient(location=":memory:")
client_ai.create_collection(
    collection_name="how_people_use_ai_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE),
)
parent_document_vectorstore_ai = QdrantVectorStore(
    collection_name="how_people_use_ai_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client_ai
)


parent_document_retriever_ai = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore_ai,
    docstore=store_ai,
    child_splitter=child_splitter_ai,
    parent_splitter=parent_splitter_ai,
)
parent_document_retriever_ai.add_documents(raw_docs_ai, ids=None)
parent_document_retrieval_chain_ai = (
    {"context": itemgetter("question") | parent_document_retriever_ai, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)


# Ensemble RAG
ensemble_retriever_ai = EnsembleRetriever(
    retrievers=[naive_retriever, bm25_retriever, multi_query_retriever, parent_document_retriever_ai],
    weights=[1/4, 1/4, 1/4, 1/4]
)

ensemble_retrieval_chain_ai = (
    {"context": itemgetter("question") | ensemble_retriever_ai, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)


# Semantic Chunking
semantic_chunker_ai = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90
)

semantic_documents_ai = semantic_chunker_ai.split_documents(raw_docs_ai)

semantic_vectorstore_ai = QdrantVectorStore.from_documents(
    semantic_documents_ai,
    embeddings,
    location=":memory:",
    collection_name="how_people_use_ai_semantic_chunks"
)

semantic_retriever_ai = semantic_vectorstore_ai.as_retriever(search_kwargs={"k" : 8})

semantic_retrieval_chain_ai = (
    {"context": itemgetter("question") | semantic_retriever_ai, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)


In [31]:
# Synthetic data generation

from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_chunks(chunks=ai_docs, testset_size=12)
dataset.to_pandas()

/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_16804/778385744.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_16804/778385744.py:8: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


Applying SummaryExtractor:   0%|          | 0/276 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/276 [00:00<?, ?it/s]

Node cc84f451-aa11-48e1-bed0-96da7ef6bf7d does not have a summary. Skipping filtering.
Node 38b44109-369f-476b-9f63-ae1a160a8e1a does not have a summary. Skipping filtering.
Node ffca7903-d55a-4257-a925-1a2d09618f52 does not have a summary. Skipping filtering.
Node 09bbff3a-7f78-4b69-976a-4dd7f3dde1e4 does not have a summary. Skipping filtering.
Node 14c911db-8195-4f7b-92cf-d2fbec93f18c does not have a summary. Skipping filtering.
Node 6e808f69-cb2d-420f-80c4-ccc732a23202 does not have a summary. Skipping filtering.
Node 446e62cd-2016-46c2-928d-c31e4e8cceea does not have a summary. Skipping filtering.
Node 8e9857ff-ec75-45b6-a407-4397f209d15b does not have a summary. Skipping filtering.
Node b60f0f9a-f4f1-4c93-8269-4fbfff06e171 does not have a summary. Skipping filtering.
Node 4067de1b-658d-4779-aef7-9eb5f1af3690 does not have a summary. Skipping filtering.
Node 68811ada-90cb-4c01-beae-6d82e84afb14 does not have a summary. Skipping filtering.
Node 464848de-5dbc-4092-b051-6c5095fcf64a d

Applying EmbeddingExtractor:   0%|          | 0/276 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,"How people use CHATGPT, what this paper say ab...",[NBER WORKING PAPER SERIES\nHOW PEOPLE USE CHA...,The NBER working paper titled HOW PEOPLE USE C...,Economic Policy Analyst,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,Was this study reviewed and approved by the Ha...,"[Harrison Satcher, Gawesha Weeratunga, Hannah...","Yes, this study was approved by Harvard IRB (I...",AI Product Research Analyst,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
2,What information is available about http://www...,[Economic Research.\nAt least one co-author ha...,Further information about the research can be ...,AI Product Research Analyst,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
3,Who is Kevin Wadman and what is his associatio...,[official NBER publications.\n© 2025 by Aaron ...,Kevin Wadman is listed as one of the authors o...,Economic Policy Analyst,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
4,How does user-level message sampling in ChatGP...,[<1-hop>\n\nat the conversation level (a conve...,User-level message sampling in ChatGPT ensures...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,what be misclassification patterns between mod...,[<1-hop>\n\nB.1.3 Conversation Topic\nAgreemen...,misclassification patterns show model and huma...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,How do regression-adjusted results alter the o...,"[<1-hop>\n\ncategories, job seniority, firm si...",Regression-adjusted results show that while ed...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,How do ChatGPT's generative AI capabilities su...,"[<1-hop>\n\nlations(3%), andData Analysis(0.4%...",ChatGPT's generative AI capabilities distingui...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,"According to Handa et al. (2025), as reference...",[<1-hop>\n\ngoals and current level of fitness...,Handa et al. (2025) report that 37% of convers...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,"How does the process of tokenization, which is...",[<1-hop>\n\nagainst another model that is trai...,Tokenization is described as a method of divid...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [60]:
import copy
import time
from ragas.metrics import LLMContextRecall, Faithfulness, ResponseRelevancy, ContextPrecision, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig
from ragas import EvaluationDataset

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

chains = {
    "naive": naive_retrieval_chain,
    "bm25": bm25_retrieval_chain,
    "multi_query": multi_query_retrieval_chain,
    "parent_document": parent_document_retrieval_chain_ai,
    "ensemble": ensemble_retrieval_chain_ai,
    "semantic": semantic_retrieval_chain_ai,
}

results = {}
for name, chain in chains.items():
    # Create a fresh copy of dataset for each chain
    dataset_copy = copy.deepcopy(dataset)  # fresh copy

    # Run chain on each question, populate eval_sample

    for test_row in dataset_copy:
        response = chain.invoke({"question": test_row.eval_sample.user_input})
        resp = response["response"]
        test_row.eval_sample.response = resp.content if hasattr(resp, "content") else resp  # or .content if AIMessage
        test_row.eval_sample.retrieved_contexts = [ctx.page_content for ctx in response["context"]]
        time.sleep(5)

    # Build EvaluationDataset, call evaluate()

    df = dataset_copy.to_pandas()

    # Replace NaN with empty strings for Ragas validation
    df['persona_name'] = df['persona_name'].fillna('')
    df['query_style'] = df['query_style'].fillna('')
    df['query_length'] = df['query_length'].fillna('')

    evaluation_dataset = EvaluationDataset.from_pandas(df)
    
    result = evaluate(
        dataset=evaluation_dataset,
        metrics=[LLMContextRecall(), Faithfulness(), ResponseRelevancy(), ContextPrecision(), ContextEntityRecall(), NoiseSensitivity()],
        llm=evaluator_llm,
        run_config=RunConfig(timeout=360)
    )

    # Store in results[name]
    results[name] = result

    # Print results
for name, result in results.items():
    print(f"Results for {name}:")
    print(result)
    print("\n")



/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_47822/1946322985.py:3: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, ResponseRelevancy, ContextPrecision, ContextEntityRecall, NoiseSensitivity
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_47822/1946322985.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, ResponseRelevancy, ContextPrecision, ContextEntityRecall, NoiseSensitivity
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_47822/1946322985.py:3: DeprecationWarning: Importing ResponseRelevanc

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


...

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[23]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[35]: TimeoutError()
Exception raised in Job[41]: TimeoutError()
Exception raised in Job[47]: TimeoutError()
Exception raised in Job[52]: TimeoutError()
Exception raised in Job[71]: TimeoutError()


Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Results for naive:
{'context_recall': 1.0000, 'faithfulness': 0.8191, 'answer_relevancy': 0.8520, 'context_precision': 0.7575, 'context_entity_recall': 0.4517, 'noise_sensitivity(mode=relevant)': 0.2018}


Results for bm25:
{'context_recall': 0.7639, 'faithfulness': 0.8153, 'answer_relevancy': 0.7782, 'context_precision': 0.5634, 'context_entity_recall': 0.2777, 'noise_sensitivity(mode=relevant)': 0.2127}


Results for multi_query:
{'context_recall': 1.0000, 'faithfulness': 0.9020, 'answer_relevancy': 0.9241, 'context_precision': 0.7132, 'context_entity_recall': 0.3730, 'noise_sensitivity(mode=relevant)': 0.3663}


Results for parent_document:
{'context_recall': 1.0000, 'faithfulness': 0.8937, 'answer_relevancy': 0.8503, 'context_precision': 0.9848, 'context_entity_recall': 0.3558, 'noise_sensitivity(mode=relevant)': 0.5656}


Results for ensemble:
{'context_recall': 1.0000, 'faithfulness': 0.9274, 'answer_relevancy': 0.9221, 'context_precision': 0.5547, 'context_entity_recall': 0.3613



#### Compiled results

| Retriever | Context Recall | Faithfulness | Answer Relevancy | Context Precision | Context Entity Recall | Noise Sensitivity* |
|-----------|----------------|--------------|------------------|-------------------|----------------------|-------------------|
| naive | 1.0000 | 0.8191 | 0.8520 | 0.7575 | 0.4517 | **0.2018** |
| bm25 | 0.7639 | 0.8153 | 0.7782 | 0.5634 | 0.2777 | 0.2127 |
| multi_query | 1.0000 | 0.9020 | 0.9241 | 0.7132 | 0.3730 | 0.3663 |
| parent_document | 1.0000 | 0.8937 | 0.8503 | **0.9848** | 0.3558 | 0.5656 |
| ensemble | 1.0000 | **0.9274** | **0.9221** | 0.5547 | 0.3613 | 0.6026 |
| semantic | 1.0000 | 0.9229 | 0.9180 | 0.7561 | **0.4490** | 0.5170 |


#### Analysis: which retriever is best for this data?


##### Key observations and insights

- **BM25 is the weakest retriever** – Lowest context recall (0.76), context precision (0.56), and context entity recall (0.28). Lexical search fits poorly with the PDF’s conceptual, research-style language.

- **Naive retrieval has a clear strength** – Highest context entity recall (0.45) and best (lowest) noise sensitivity (0.20), making it the most robust to irrelevant context. It trails on faithfulness and answer relevancy.

- **Parent document has the best context precision (0.98)** – It returns highly relevant chunks with little noise. The child–parent structure helps keep retrieval focused.

- **Ensemble leads on faithfulness (0.93) and answer relevancy (0.92)** – Combining multiple retrieval strategies yields more faithful answers. However, it has the worst (highest) noise sensitivity (0.60), so it is least robust to irrelevant context.

- **Semantic retrieval is strong and balanced** – High faithfulness (0.92), answer relevancy (0.92), and context entity recall (0.45). It performs well without the extra complexity of ensemble or parent-document.

- **Multi-query, parent document, ensemble, and semantic all reach full context recall (1.0)** – They retrieve the needed information; differences show up in precision, faithfulness, and robustness.



##### Analysis

For a Q&A system over *howpeopleuseai.pdf* (research-style document):

1. **Faithfulness** – Answers must be grounded in the document; hallucinations are unacceptable.
2. **Answer relevancy** – Answers must directly address the user’s question.
3. **Noise sensitivity** (lower is better) – The system should remain reliable when some irrelevant context is present.
4. **Context precision** – Retrieving highly relevant chunks avoids wasting context and improves answer quality.

Context recall is less critical here because most methods already reach 1.0. Context entity recall is useful but secondary for general Q&A.



##### Conclusion

**Ensemble** is the best fit for this use case: it leads on faithfulness and answer relevancy, which are the most important for a reliable Q&A system over this document. **Parent document** is a strong alternative if we want maximum context precision and simpler retrieval logic. **Naive** and **BM25** have the best (lowest) noise sensitivity and are most robust to irrelevant context.

These numbers might have been affected by the timeouts. So, I would like to run the evaluations again with new run_config and see if the results change.

In [33]:
import copy
import time
from ragas.metrics import LLMContextRecall, Faithfulness, ResponseRelevancy, ContextPrecision, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig
from ragas import EvaluationDataset

run_config_new=RunConfig(timeout=600, max_workers=8)

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

chains = {
    "naive": naive_retrieval_chain,
    "bm25": bm25_retrieval_chain,
    "multi_query": multi_query_retrieval_chain,
    "parent_document": parent_document_retrieval_chain_ai,
    "ensemble": ensemble_retrieval_chain_ai,
    "semantic": semantic_retrieval_chain_ai,
}

results_new = {}
for name, chain in chains.items():
    # Create a fresh copy of dataset for each chain
    dataset_copy = copy.deepcopy(dataset)  # fresh copy

    # Run chain on each question, populate eval_sample

    for test_row in dataset_copy:
        response = chain.invoke({"question": test_row.eval_sample.user_input})
        resp = response["response"]
        test_row.eval_sample.response = resp.content if hasattr(resp, "content") else resp  # or .content if AIMessage
        test_row.eval_sample.retrieved_contexts = [ctx.page_content for ctx in response["context"]]
        time.sleep(5)

    # Build EvaluationDataset, call evaluate()

    df = dataset_copy.to_pandas()

    # Replace NaN with empty strings for Ragas validation
    df['persona_name'] = df['persona_name'].fillna('')
    df['query_style'] = df['query_style'].fillna('')
    df['query_length'] = df['query_length'].fillna('')

    evaluation_dataset = EvaluationDataset.from_pandas(df)
    
    result_new = evaluate(
        dataset=evaluation_dataset,
        metrics=[LLMContextRecall(), Faithfulness(), ResponseRelevancy(), ContextPrecision(), ContextEntityRecall(), NoiseSensitivity()],
        llm=evaluator_llm,
        run_config=run_config_new
    )

    # Store in results[name]
    results_new[name] = result_new

    # Print results
for name, result in results_new.items():
    print(f"Results for {name}:")
    print(result)
    print("\n")


/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_16804/3457783465.py:3: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, ResponseRelevancy, ContextPrecision, ContextEntityRecall, NoiseSensitivity
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_16804/3457783465.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, ResponseRelevancy, ContextPrecision, ContextEntityRecall, NoiseSensitivity
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_16804/3457783465.py:3: DeprecationWarning: Importing ResponseRelevanc

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


...

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[5]: TimeoutError()


Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Results for naive:
{'context_recall': 0.9167, 'faithfulness': 0.7381, 'answer_relevancy': 0.8698, 'context_precision': 0.8039, 'context_entity_recall': 0.5847, 'noise_sensitivity(mode=relevant)': 0.2436}


Results for bm25:
{'context_recall': 0.7778, 'faithfulness': 0.8402, 'answer_relevancy': 0.9529, 'context_precision': 0.6990, 'context_entity_recall': 0.5606, 'noise_sensitivity(mode=relevant)': 0.1932}


Results for multi_query:
{'context_recall': 0.9583, 'faithfulness': 0.8377, 'answer_relevancy': 0.8661, 'context_precision': 0.7271, 'context_entity_recall': 0.5835, 'noise_sensitivity(mode=relevant)': 0.2378}


Results for parent_document:
{'context_recall': 0.9167, 'faithfulness': 0.8318, 'answer_relevancy': 0.9494, 'context_precision': 0.9444, 'context_entity_recall': 0.6203, 'noise_sensitivity(mode=relevant)': 0.3422}


Results for ensemble:
{'context_recall': 0.9167, 'faithfulness': 0.8608, 'answer_relevancy': 0.8758, 'context_precision': 0.5378, 'context_entity_recall': 0.6435


#### Compiled results

| Retriever | Context Recall | Faithfulness | Answer Relevancy | Context Precision | Context Entity Recall | Noise Sensitivity |
|-----------|----------------|--------------|------------------|-------------------|------------------------|-------------------|
| naive | 0.9167 | 0.7381 | 0.8698 | 0.8039 | 0.5847 | 0.2436 |
| bm25 | 0.7778 | 0.8402 | 0.9529 | 0.6990 | 0.5606 | **0.1932** |
| multi_query | 0.9583 | 0.8377 | 0.8661 | 0.7271 | 0.5835 | 0.2378 |
| parent_document | 0.9167 | 0.8318 | 0.9494 | **0.9444** | 0.6203 | 0.3422 |
| ensemble | 0.9167 | **0.8608** | 0.8758 | 0.5378 | 0.6435 | 0.3035 |
| semantic | **0.9583** | 0.8219 | 0.9458 | 0.9188 | **0.6942** | 0.3029 |

#### Analysis: Which retriever is best for this data?

##### Observations and insights

- **BM25** has the lowest context recall (0.78) and highest answer relevancy (0.95). Lexical search misses some relevant chunks but often returns highly relevant ones when it matches.
- **Naive** has the lowest faithfulness (0.74) and lowest context entity recall (0.58). Simple retrieval leaves more room for hallucination and misses entities.
- **Parent document** has the best context precision (0.94) and strong answer relevancy (0.95). Child–parent retrieval keeps context focused and relevant.
- **Ensemble** has the highest faithfulness (0.86) and good context entity recall (0.64), but the lowest context precision (0.54). Combining retrievers improves grounding but adds noisy chunks.
- **Semantic** leads on context recall (0.96) and context entity recall (0.69), with high context precision (0.92). Semantic chunking and retrieval fit this document well.
- **Multi-query** ties for best context recall (0.96) and is balanced across metrics.



##### Analysis

For a Q&A system over *howpeopleuseai.pdf*:

- **Faithfulness** – Answers must be grounded in the document; ensemble is strongest.
- **Answer relevancy** – Answers must address the question; BM25, parent document, and semantic are all strong.
- **Context precision** – Retrieved chunks should be relevant; parent document and semantic are best.
- **Noise sensitivity** – Robustness to irrelevant context; parent document is most robust.



##### Conclusion

- **Best overall** – **Semantic** is the strongest overall: high context recall, high context precision, best context entity recall, and good faithfulness and answer relevancy.
- **Best precision** – **Parent document** is best when precision of retrieved context is the main concern.
- **Best faithfulness** – **Ensemble** is best when minimizing hallucination is the priority.
- **Weakest** – **Naive** is weakest on faithfulness and entity recall; **BM25** is weakest on context recall.

**Recommendation:** For this Q&A use case, **semantic** is the best default choice; **parent document** is a strong alternative when precision of retrieved context is the main concern.



#### How Conclusions Changed Between the Runs

- **Previous:** Ensemble was best overall (faithfulness, answer relevancy, noise sensitivity). Parent document was best for context precision. Semantic was strong and balanced.
- **New:** Semantic is best overall (context recall, context precision, context entity recall). Parent document is still best for context precision. Ensemble is still best for faithfulness but weaker on noise sensitivity.

**Changes in rank:**

1. **Semantic** moves up from “strong and balanced” to the best overall.
2. **Ensemble** stays strong on faithfulness but loses its advantage on noise sensitivity.
3. **Parent document** remains best for precision.
4. **BM25** improves on answer relevancy and entity recall.



##### Why the differences?

- Different synthetic data. (I had to restart the kernel and regenerate)
- Possible timeouts / skipped samples in the previous run.



#### Updated conclusion

- **Best overall:** Semantic (context recall, precision, entity recall).
- **Best precision:** Parent document.
- **Best faithfulness:** Ensemble.
- **Most improved:** BM25 (answer relevancy, entity recall).

The relative rankings are similar, but the new run is more favorable to semantic and BM25.

I could also address this limitation: "LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.", maybe by using Open AI llm_factory instead of the LangchainLLMWrapper.  I got into some rate limits and weird kernel issues, trying this. So, that's for later. :) 

![Overall](./data/ls-overall.png)

- Reliability – 0% error rate across all traces.
- Typical latency – Median 5.78s, so most runs are reasonably fast.
- Tail latency – P99 at 75s shows some runs are much slower.
- Usage – ~6.2M tokens and ~$5.39 in cost for the week.

#### Langsmith - Key observations


##### Cost and tokens

![cost and tokens](./data/ls-cost.png)

- **Cost distribution** – P50 cost per trace is near $0; P99 is much higher. A small fraction of traces drives most of the cost.
- **Cost vs. tokens** – Cost and token usage are strongly correlated; token usage is the main cost driver.
- **Ragas evaluation** – One evaluation run: ~$0.47, ~686K tokens, ~9.7 minutes. Evaluation dominates cost and latency.

##### Latency

![latency](./data/ls-latency.png)

- **P50 vs. P99** – Median latency is very low; P99 shows large spikes (up to ~20 minutes). Most calls are fast; a few are extremely slow.
- **RAG vs. evaluation** – Single RAG queries: ~2–8 seconds, ~$0.0005–0.001. Ragas evaluation: ~9.7 minutes, ~$0.47. Evaluation is far slower and more expensive than inference.

##### Errors and reliability

- **Error rate** – There is a period with many errors (likely rate limiting), followed by a period with mostly successful runs.
- **Slow vs. failed calls** – High P99 latency and high error rate occur in different periods; slow calls are not the same as failed ones.

![traces](./data/ls-traces.png)

##### Summary

- **Evaluation** – Ragas evaluation is the main cost and latency source.
- **Inference** – Individual RAG queries are cheap and fast.
- **Distribution** – Most traces are cheap and fast; a few are expensive and slow.
- **Errors** – Early errors suggest rate limiting; later runs are more stable.